# Urdu QA: train XLM-RoBERTa on the paragraph-level split

Runs the whole re-evaluation on a free Colab GPU:

1. clones [`zarekasif/urdu-qa`](https://github.com/zarekasif/urdu-qa) and installs it,
2. reads the translated Urdu SQuAD CSV from your Google Drive,
3. builds a train/dev/test split by paragraph (no leakage),
4. runs a 2-minute smoke test, then trains XLM-RoBERTa base,
5. scores dev with standard SQuAD Exact Match and F1 and saves everything to Drive.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, and make sure your dataset files are in Google Drive (set the folder below).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/zarekasif/urdu-qa.git
%cd urdu-qa
!pip install -q -e ".[train]"

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in your Drive that holds the dataset files (change if needed).
DATA_DIR = '/content/drive/MyDrive/FYP/DATASET'

# The full dataset CSV (recommended) ...
INPUTS = ['URDU TRANSLATED DATASET.csv']
# ... or the old pre-split JSON files, which are pooled and re-split by paragraph:
# INPUTS = ['TRAIN90.json', 'VALIDATION5.json', 'TEST5.json']

RESULTS_DIR = '/content/drive/MyDrive/urdu-qa-results'

import os
print(sorted(os.listdir(DATA_DIR)))
INPUT_ARGS = ' '.join(f'"{DATA_DIR}/{f}"' for f in INPUTS)

In [ ]:
!python -m urduqa.prepare {INPUT_ARGS} data/processed/all.jsonl
!python -m urduqa.split data/processed/all.jsonl data/processed --seed 13
# Expected on the 2022 data: kept 40273, train 32290 / dev 3959 / test 4024, leakage 0.0%


## Smoke test (about 2 minutes)

Trains on 200 questions for one epoch. The score will be near zero; the point is to check
that the pipeline runs end to end before spending an hour on the full run.

In [ ]:
!python -m urduqa.train_extractive --data-dir data/processed --out-dir outputs/smoke \
    --max-train-samples 200 --max-eval-samples 100 --epochs 1

## Full run: XLM-RoBERTa base, 2 epochs (about 30–60 minutes on a T4)

If Colab runs out of GPU memory, add `--batch-size 8`.

In [ ]:
!python -m urduqa.train_extractive --data-dir data/processed --out-dir outputs/xlmr-base \
    --model xlm-roberta-base --epochs 2 --lr 3e-5 --batch-size 16

## Save results to Drive

In [ ]:
import json, shutil, os
os.makedirs(RESULTS_DIR, exist_ok=True)
for f in ['metrics.json', 'predictions.json']:
    shutil.copy(f'outputs/xlmr-base/{f}', f'{RESULTS_DIR}/xlmr-base-{f}')
print(json.dumps(json.load(open('outputs/xlmr-base/metrics.json')), indent=2))

Paste the printed metrics back into the chat. The fine-tuned model is in
`outputs/xlmr-base/model` (about 1 GB); copy it to Drive too if you want to keep it.